In [5]:
from datasets import Dataset, DatasetDict
import torch
import pickle
import os
from ICL.datasets.RHM import RandomHierarchyModel  # Your existing class


def create_hf_dataset_from_rhm(config_list, samples_per_config=1000, output_dir='./hf_datasets'):
    """
    Use existing RHM class to generate HuggingFace compatible dataset
    """
    
    all_input_ids = []
    all_labels = []  
    all_task_ids = []
    all_config_L = []
    all_config_m = []
    all_lengths = []
    all_rules = {}
    
    for task_id, (L, m) in enumerate(config_list):
        print(f"Generating task {task_id}: L={L}, m={m}")
        
        # Use existing RHM class exactly as-is
        rhm = RandomHierarchyModel(
            num_features=32,        # vocabulary size
            num_classes=10,         # number of classes  
            num_synonyms=m,         # multiplicity
            tuple_size=2,           # size of low-level representations
            num_layers=L,           # number of levels in hierarchy
            seed_rules=task_id,     # different rules per task
            seed_sample=42,         # fixed for reproducibility
            train_size=samples_per_config,
            test_size=0,
            input_format='long',    # gets integer sequences
            replacement=True
        )
        
        # Extract data from RHM object
        sequences = rhm.features   # Shape: [samples_per_config, sequence_length]
        labels = rhm.labels        # Shape: [samples_per_config]
        rules = rhm.rules          # Production rules dictionary
        
        # Convert to lists (HuggingFace prefers lists)
        sequences_list = sequences.tolist()
        labels_list = labels.tolist()
        
        # Store rules for this task
        all_rules[task_id] = rules
        
        # Add to combined dataset
        all_input_ids.extend(sequences_list)
        all_labels.extend(labels_list)
        all_task_ids.extend([task_id] * len(sequences_list))
        all_config_L.extend([L] * len(sequences_list))
        all_config_m.extend([m] * len(sequences_list))
        all_lengths.extend([len(seq) for seq in sequences_list])
    
    # Create HuggingFace Dataset
    dataset_dict = {
        'input_ids': all_input_ids,    # Raw integer sequences
        'labels': all_labels,          # Classification targets
        'task_id': all_task_ids,       # Which RHM configuration  
        'config_L': all_config_L,      # Hierarchy depth
        'config_m': all_config_m,      # Multiplicity
        'length': all_lengths          # Sequence length
    }
    
    dataset = Dataset.from_dict(dataset_dict)
    
    # Save dataset
    os.makedirs(output_dir, exist_ok=True)
    dataset.save_to_disk(f"{output_dir}/mixed_rhm_dataset")
    
    # Save metadata
    metadata = {
        'configs': [{'task_id': i, 'L': L, 'm': m} for i, (L, m) in enumerate(config_list)],
        'vocab_size': 32,
        'num_classes': 10, 
        'tuple_size': 2,
        'samples_per_config': samples_per_config,
        'rules': all_rules
    }
    
    with open(f"{output_dir}/metadata.pkl", 'wb') as f:
        pickle.dump(metadata, f)
    
    print(f"Dataset saved to {output_dir}")
    print(f"Total samples: {len(dataset)}")
    print(f"Sequence length range: {min(dataset['length'])} - {max(dataset['length'])}")
    
    return dataset, metadata

# Usage - Generate mixed RHM dataset
config_list = [(2,4), (2,8), (3,4), (3,8), (3,16)]
dataset, metadata = create_hf_dataset_from_rhm(config_list, samples_per_config=2000)

ModuleNotFoundError: No module named 'ICL'